In [1]:
import os, pickle, json
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
import spacy
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import numpy as np

In [2]:
# Leer chunks, está en excel
chunks_etiquetados = pd.read_excel("../data/results/chunks_etiquetados_binario.xlsx")
chunks_etiquetados.head()

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.411230,0.368818,0.363447,0.354240,0.378154,0.343397,0.387251,...,0.392998,0.423331,0.337017,0.352550,0.328395,0.394268,0.393815,0.389038,"[('ninguna', 0)]",0
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...",0.331604,0.304309,0.346682,0.315253,0.344347,0.323262,0.344607,...,0.318409,0.378231,0.317467,0.368125,0.306375,0.381964,0.337864,0.306510,"[('ninguna', 0)]",0
2,2,1,los acuerdos con las Farc. Anunció que no prom...,0.356805,0.311163,0.303825,0.287517,0.329738,0.290927,0.332603,...,0.292858,0.405040,0.291387,0.322885,0.297822,0.338846,0.294842,0.318955,"[('ninguna', 0)]",0
3,3,1,moratoria en la explotación tipo fracking. Y f...,0.435032,0.397881,0.372960,0.368983,0.415538,0.357000,0.410173,...,0.392092,0.445708,0.370653,0.367256,0.341832,0.404120,0.375238,0.397525,"[('ninguna', 0)]",0
4,0,2,Las interpretaciones de la historia sirven com...,0.352643,0.367811,0.375234,0.357391,0.368184,0.347770,0.392265,...,0.368177,0.428733,0.353389,0.405911,0.371109,0.412153,0.397781,0.339795,"[('ninguna', 0)]",0


In [3]:
chunks_ciencias = chunks_etiquetados[chunks_etiquetados["etiqueta_ciencia"] == 1]
chunks_ciencias.shape

(7267, 22)

In [4]:
# Leer embeddings gemma
embeddings = np.load(r"../data/processed/chunk_embeddings.npy")

In [5]:
# máscara booleana
mask_ciencias = chunks_etiquetados["etiqueta_ciencia"] == 1

# filtrar embeddings con la misma máscara
embeddings_ciencias = embeddings[mask_ciencias.values]

embeddings_ciencias.shape


(7267, 768)

# Stopwords

In [6]:
# Cargar modelo pequeño de spaCy (rápido)
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

# Stopwords en español desde spaCy
spanish_stopwords = nlp.Defaults.stop_words

# BERTopic

In [14]:
documents = chunks_ciencias['texto_chunk'].tolist()

In [15]:
vectorizer_model = CountVectorizer(
    stop_words= list(spanish_stopwords),
    max_features=5000,   
    min_df=5,
    ngram_range=(1, 1),
    strip_accents=None
)

In [16]:
def get_topic_words(topic_model, top_n=10):
    topics_words = []

    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue  # excluimos outliers

        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)

    return topics_words

In [17]:
# Función que junta ambos métodos

def compute_bertopic_coherence(
    topic_model,
    documents,
    vectorizer_model,
    top_n_words=10
):
    analyzer = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]

    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)

    corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    topics_words = get_topic_words(topic_model, top_n=top_n_words)

    coherence_model = CoherenceModel(
        topics=topics_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        corpus=corpus,
        coherence='c_v'
    )

    return coherence_model.get_coherence(), coherence_model.get_coherence_per_topic()

In [19]:
nr_topics_values = [10, 20, 30, 40, 60, "auto"]
results = []

for nr in nr_topics_values:
    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=20,
        calculate_probabilities=False,
        verbose=True,
        language="spanish"
    )

    topics, probs = topic_model.fit_transform(
        documents,
        embeddings=embeddings_ciencias   # embeddings Gemma
    )

    new_topics  = topic_model.reduce_outliers(
        documents,
        topics,
        strategy="c-tf-idf",
        embeddings=embeddings_ciencias
        )

    topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence,_ = compute_bertopic_coherence(topic_model, documents, vectorizer_model)

    n_outliers = sum(1 for t in topic_model.topics_ if t == -1)


    results.append({
        "nr_topics": nr,
        "n_topics_final": len([t for t in topic_model.get_topics() if t != -1]),
        "coherence_c_v": coherence,
        "%_outliers": n_outliers / len(documents)
    })


2026-01-22 01:24:32,600 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-22 01:24:48,081 - BERTopic - Dimensionality - Completed ✓
2026-01-22 01:24:48,083 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-22 01:24:48,349 - BERTopic - Cluster - Completed ✓
2026-01-22 01:24:48,364 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-22 01:24:49,351 - BERTopic - Representation - Completed ✓
2026-01-22 01:24:49,351 - BERTopic - Topic reduction - Reducing number of topics
2026-01-22 01:24:49,386 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-22 01:24:49,965 - BERTopic - Representation - Completed ✓
2026-01-22 01:24:49,965 - BERTopic - Topic reduction - Reduced number of topics from 29 to 10
2026-01-22 01:24:50,380 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure tha

In [20]:
df_results = pd.DataFrame(results)
df_results.sort_values("coherence_c_v", ascending=False)

,nr_topics,n_topics_final,coherence_c_v,%_outliers
1,20,19,0.695491,0.0
3,40,28,0.695162,0.0
5,auto,26,0.694176,0.0
2,30,28,0.681233,0.0
4,60,27,0.678842,0.0
0,10,9,0.622583,0.0


In [36]:
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=20,
    min_topic_size=20,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)
topics, probs = topic_model.fit_transform(
    documents,
    embeddings=embeddings_ciencias   # embeddings Gemma
)

new_topics  = topic_model.reduce_outliers(
        documents,
        topics,
        strategy="c-tf-idf",
        embeddings=embeddings_ciencias
        )

topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)


2026-01-22 02:21:58,707 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-22 02:22:00,523 - BERTopic - Dimensionality - Completed ✓
2026-01-22 02:22:00,523 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-22 02:22:00,736 - BERTopic - Cluster - Completed ✓
2026-01-22 02:22:00,736 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-01-22 02:22:01,307 - BERTopic - Representation - Completed ✓
2026-01-22 02:22:01,307 - BERTopic - Topic reduction - Reducing number of topics
2026-01-22 02:22:01,323 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-22 02:22:01,870 - BERTopic - Representation - Completed ✓
2026-01-22 02:22:01,870 - BERTopic - Topic reduction - Reduced number of topics from 29 to 20
2026-01-22 02:22:02,314 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure tha

In [37]:
tm_res = topic_model.get_topic_info()

In [38]:
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
topic_ids = [
    topic_id
    for topic_id in topic_model.get_topics().keys()
    if topic_id != -1
]
coherence_df = pd.DataFrame({
    "Topic": topic_ids,
    "Coherence_c_v": coh_per_topic
})

coherence_table = (
    tm_res[tm_res["Topic"] != -1]
    .merge(coherence_df, on="Topic", how="left")
)

In [39]:
total_docs = tm_res["Count"].sum()

In [40]:
top = (
    tm_res
    .sort_values("Count", ascending=False)
    .head(10)
    .copy()
)
top["Keywords"] = top["Representation"].apply(
    lambda words: ", ".join(words)
)


In [41]:
top["Percentage"] = (top["Count"] / total_docs * 100).round(2)

top = top.merge(
    coherence_df,
    on="Topic",
    how="left"
)


In [42]:
final_table = top[[
    "Topic",
    #"Count",
    "Coherence_c_v",
    "Percentage",
    "Keywords"
]]

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,1,0.493785,25.62,"política, gobierno, paz, país, seguridad, violencia, político, políticos, justicia, colombia"
1,0,0.460526,21.69,"salud, mundo, vida, pandemia, tiempo, ciencia, personas, virus, años, crisis"
2,3,0.574302,11.42,"economía, desarrollo, crecimiento, recursos, país, sector, gobierno, política, fiscal, innovación"
3,2,0.782987,9.61,"educación, universidad, formación, universidades, estudiantes, nacional, calidad, investigación, conocimiento, superior"
4,4,0.784604,6.56,"ambiental, ambientales, conservación, ambiente, biodiversidad, sostenible, desarrollo, deforestación, bosques, ecosistemas"
5,5,0.799532,4.79,"climático, cambio, planeta, global, calentamiento, incendios, mundo, climática, clima, temperatura"
6,8,0.504077,2.79,"datos, información, encuestas, acceso, transparencia, resultados, personas, medios, ejemplo, pública"
7,7,0.686045,2.75,"agua, río, aguas, lluvia, ríos, cauca, barrios, obras, bogotá, país"
8,9,0.919135,2.02,"energía, combustibles, fósiles, energías, carbón, renovables, solar, energética, petróleo, gas"
9,10,0.724657,1.83,"mujeres, hombres, mujer, género, femenino, violencia, personas, vida, femenina, sexo"


In [43]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", None)

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,1,0.493785,25.62,"política, gobierno, paz, país, seguridad, violencia, político, políticos, justicia, colombia"
1,0,0.460526,21.69,"salud, mundo, vida, pandemia, tiempo, ciencia, personas, virus, años, crisis"
2,3,0.574302,11.42,"economía, desarrollo, crecimiento, recursos, país, sector, gobierno, política, fiscal, innovación"
3,2,0.782987,9.61,"educación, universidad, formación, universidades, estudiantes, nacional, calidad, investigación, conocimiento, superior"
4,4,0.784604,6.56,"ambiental, ambientales, conservación, ambiente, biodiversidad, sostenible, desarrollo, deforestación, bosques, ecosistemas"
5,5,0.799532,4.79,"climático, cambio, planeta, global, calentamiento, incendios, mundo, climática, clima, temperatura"
6,8,0.504077,2.79,"datos, información, encuestas, acceso, transparencia, resultados, personas, medios, ejemplo, pública"
7,7,0.686045,2.75,"agua, río, aguas, lluvia, ríos, cauca, barrios, obras, bogotá, país"
8,9,0.919135,2.02,"energía, combustibles, fósiles, energías, carbón, renovables, solar, energética, petróleo, gas"
9,10,0.724657,1.83,"mujeres, hombres, mujer, género, femenino, violencia, personas, vida, femenina, sexo"
